# 03-area: stage 1 on the full area

Stage 1 runs the chain on the whole area with the code in `src/catchment/`. This first section is the **snapping fix**
(#14): the before and after, on the slice and on the area. The later sections (the area demand map, the full-area
solve, the spot-check) come in other issues.

Stage 0.5 found about one short pair in seven riding more than 300 m out and back at one end, from snapping onto the far
carriageway of a divided road. `catchment-build reach` now snaps each candidate site and each hex centre to two roads
and keeps the fastest of the four times for a pair; the 17-min cap applies after the minimum. This notebook reads what
`catchment-build` wrote (the after), rebuilds the one-snap table in the same session (the before), and compares them.

In [ ]:
import resource
import tempfile
import time
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyproj import Transformer
from shapely.geometry import box

from catchment import assumptions
from catchment.area import load_area
from catchment.fetch import DATA
from catchment.reach import build_reach, served_osrm

FIGS = DATA.parent / "notebooks" / "figures"
FIGS.mkdir(parents=True, exist_ok=True)
MAX_MIN = assumptions.MAX_PAIR_MIN
DEFAULT_MIN = (assumptions.PROMISE - assumptions.PICK_AND_PACK) / assumptions.MULTIPLIER   # 5.0 free-flow minutes
area = load_area()
slice_poly = box(77.005, 28.405, 77.105, 28.480).intersection(area)   # the 0.5 slice
demand = pd.read_parquet(DATA / "demand-r9.parquet")
cand_cmd = pd.read_parquet(DATA / "candidates-r9-x30.parquet")        # catchment-build's outputs (the after)
table_cmd = pd.read_parquet(DATA / "table-r9-x30.parquet")
pts = gpd.read_parquet(DATA / "overture_places.parquet")
places = pd.DataFrame({"lon": pts.geometry.x, "lat": pts.geometry.y, "operating_status": pts.operating_status})   # as build.py loads them
to_utm = Transformer.from_crs(4326, 32643, always_xy=True)   # metres, for distances along the route

## Assumptions

One row of `assumptions.table()` is new in this section. **Assumption** means a labelled choice with no external
source.

In [ ]:
rows = assumptions.table()
pd.set_option("display.max_colwidth", None)
display(rows[rows.iloc[:, 0] == "Second road for snapping"].style.hide(axis="index").set_properties(**{"text-align": "left"}))

## Before and after, measured in one session

Inside one `served_osrm` block the table is built twice with the same function: `snaps=1` (the nearest road only, the
#13 build) and `snaps=2` (the fix). The two-snap table must equal the one `catchment-build` wrote; that check is what
lets the rest of the notebook read the command's output. The block also snaps every candidate site and hex centre to two
roads (to count how many got a distinct second road) and runs the out-and-back sample from the slice section below.

In [ ]:
def route_pair(url, src, dst):
    """One /route between two snapped points: minutes, path m while within 100 m of the start, path m after first
    reaching 100 m of the end (the 01-reach method). > 300 m at an end counts as out-and-back."""
    import requests
    rt = requests.get(f"{url}/route/v1/driving/{src[0]:.6f},{src[1]:.6f};{dst[0]:.6f},{dst[1]:.6f}",
                      params={"overview": "full", "geometries": "geojson"}, timeout=300).json()
    assert rt["code"] == "Ok", rt
    (ax, ay), (bx, by) = (to_utm.transform(*w["location"]) for w in rt["waypoints"])
    x, y = to_utm.transform(*np.array(rt["routes"][0]["geometry"]["coordinates"]).T)
    along = np.r_[0, np.cumsum(np.hypot(np.diff(x), np.diff(y)))]
    leave = along[np.where(np.hypot(x - ax, y - ay) <= 100)[0].max()]
    arrive = along[-1] - along[np.where(np.hypot(x - bx, y - by) <= 100)[0].min()]
    return rt["routes"][0]["duration"] / 60, leave, arrive

def in_poly(df, poly):
    return gpd.GeoSeries(gpd.points_from_xy(df.lon, df.lat), index=df.index).within(poly)

def timed(snaps, router):
    t0 = time.perf_counter()
    c, t = build_reach(area, demand, places, 9, 30, router, snaps=snaps)
    return c, t, time.perf_counter() - t0

area_hexes = demand[in_poly(demand, area)]
slice_hexes = area_hexes[in_poly(area_hexes, slice_poly)]
slice_cands = cand_cmd[in_poly(cand_cmd, slice_poly)]

with served_osrm(DATA) as router:
    cand_before, table_before, secs_before = timed(1, router)
    cand_after, table_after, secs_after = timed(2, router)
    c_lon, c_lat, _ = router.snap(cand_cmd.lon.to_numpy(), cand_cmd.lat.to_numpy(), k=2)
    h_lon, h_lat, _ = router.snap(area_hexes.lon.to_numpy(), area_hexes.lat.to_numpy(), k=2)
    # out-and-back sample: 200 slice pairs with BEFORE minutes <= 6, candidate != hex, seed 42
    in_slice = table_before.candidate.isin(slice_cands.index) & table_before.hex.isin(slice_hexes.index)
    short = table_before[in_slice & (table_before.minutes <= 6) & (table_before.candidate != table_before.hex)]
    c_pos = {h: i for i, h in enumerate(cand_cmd.index)}
    h_pos = {h: i for i, h in enumerate(area_hexes.index)}
    samples = []
    for _, p in short.sample(200, random_state=42).iterrows():
        i, j = c_pos[p.candidate], h_pos[p.hex]
        c_pts = [(c_lon[i, a], c_lat[i, a]) for a in range(2)]
        h_pts = [(h_lon[j, b], h_lat[j, b]) for b in range(2)]
        before = route_pair(router.url, c_pts[0], h_pts[0])
        after = min((route_pair(router.url, c, h) for c in c_pts for h in h_pts), key=lambda r: r[0])
        samples.append({"candidate": p.candidate, "hex": p.hex, "table min before": p.minutes,
                        "before min": before[0], "before leave m": before[1], "before arrive m": before[2],
                        "after min": after[0], "after leave m": after[1], "after arrive m": after[2]})
peak_bytes = router.peak_bytes   # osrm-routed's peak over both builds, the snaps and the sample; read when the block stopped
samples = pd.DataFrame(samples)

# the notebook must read what the command builds
assert list(cand_after.index) == list(cand_cmd.index)
key = ["candidate", "hex"]
a, b = table_after.sort_values(key).reset_index(drop=True), table_cmd.sort_values(key).reset_index(drop=True)
assert len(a) == len(b) and (a[key] == b[key]).all().all() and np.allclose(a.minutes, b.minutes, atol=1e-4), "snaps=2 differs from catchment-build"
print("snaps=2 table equals catchment-build's:", len(a), "pairs")

### Full area

Pairs, table size on disk, build time and memory, before and after. The before table is written to a temporary file to
measure its size; the after size is the file `catchment-build` wrote. The times are `build_reach` alone (OSRM already
serving), so a little under `catchment-build`'s wall time.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    before_path = Path(tmp) / "table-before.parquet"
    table_before.to_parquet(before_path)
    size_before = before_path.stat().st_size
size_after = (DATA / "table-r9-x30.parquet").stat().st_size
full = pd.DataFrame({
    "before (1 snap)": [len(table_before), (table_before.minutes <= DEFAULT_MIN).sum(), size_before / 1e6, secs_before],
    "after (2 snaps)": [len(table_after), (table_after.minutes <= DEFAULT_MIN).sum(), size_after / 1e6, secs_after],
}, index=[f"pairs within {MAX_MIN} min", f"pairs within {DEFAULT_MIN:.1f} min (default scenario)", "table on disk, MB", "build_reach time, s"])
full["change"] = full["after (2 snaps)"] / full["before (1 snap)"] - 1
display(full.round(2))
print(f"osrm-routed peak (both builds, snaps and sample): {peak_bytes / 1e6:,.0f} MB")
print(f"this notebook process's peak RSS: {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:,.0f} MB (holds both tables and the places)")

# second road: how many points got a snap point different from their nearest road's
def distinct(lon, lat):
    return ((lon[:, 0] != lon[:, 1]) | (lat[:, 0] != lat[:, 1]))
d_c, d_h = distinct(c_lon, c_lat), distinct(h_lon, h_lat)
print(f"distinct second road: {d_c.sum():,} of {len(d_c):,} candidate sites ({d_c.mean():.1%}), "
      f"{d_h.sum():,} of {len(d_h):,} hex centres ({d_h.mean():.1%})")

### Slice

The same slice as 0.5. Pairs whose candidate site and hex both have centres in the slice, outer-joined before and after
on (candidate, hex). A pair moved if its time changed by more than 0.01 min; the histogram is over moved pairs that were
already within 17 min before.

In [ ]:
def slice_only(t):
    return t[t.candidate.isin(slice_cands.index) & t.hex.isin(slice_hexes.index)]

j = slice_only(table_before).merge(slice_only(table_after), on=key, how="outer", suffixes=("_b", "_a"))
both = j.dropna(subset=["minutes_b", "minutes_a"])
saved = both.minutes_b - both.minutes_a
moved = saved[saved.abs() > 0.01]
faster, slower = moved[moved > 0], moved[moved < 0]
new_17 = j.minutes_b.isna().sum()
new_5 = ((j.minutes_a <= DEFAULT_MIN) & ~(j.minutes_b <= DEFAULT_MIN)).sum()
print(f"slice pairs: {len(slice_only(table_before)):,} before, {len(slice_only(table_after)):,} after; only before: {j.minutes_a.isna().sum()}")
print(f"moved (|d| > 0.01 min): {len(moved):,} of {len(both):,} ({len(moved) / len(both):.1%}); slower: {len(slower)} (worst {slower.min() if len(slower) else 0:.3f} min)")
print(f"minutes saved over faster pairs: median {faster.median():.2f}, 90th pct {faster.quantile(.9):.2f}, max {faster.max():.2f}")
print(f"faster by more than 0.5 min: {(moved > 0.5).sum():,} ({(moved > 0.5).sum() / len(both):.1%}); by more than 1 min: {(moved > 1).sum():,} ({(moved > 1).sum() / len(both):.1%})")
print(f"newly within {MAX_MIN} min: {new_17:,}; newly within {DEFAULT_MIN:.1f} min: {new_5:,}")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(faster, bins=40, color="#3b6ea5")
ax.set_xlabel("minutes saved by the fastest of four (before - after)")
ax.set_ylabel("pairs")
ax.set_title(f"Slice: {len(faster):,} pairs got faster ({len(faster) / len(both):.1%} of {len(both):,})")
fig.tight_layout()
fig.savefig(FIGS / "03-snapping.png", dpi=150)
plt.show()

![Minutes saved on the slice's moved pairs](figures/03-snapping.png)

### Out-and-back on short pairs

200 slice pairs (seed 42) with free-flow time at most 6 min before and candidate site different from hex. Before: one
`/route` from the candidate's nearest snap to the hex's nearest snap. After: the fastest of the four `/route` calls
between the two snaps of each end. Path while still within 100 m of the start, and after first reaching 100 m of the
end, as in 01-reach; more than 300 m counts as out-and-back.

In [ ]:
for s in ("before", "after"):
    samples[f"{s} path m"] = samples[[f"{s} leave m", f"{s} arrive m"]].max(axis=1)
rows = {}
for s in ("before", "after"):
    rows[s] = {"pairs": len(samples),
               "> 300 m at either end": int((samples[f"{s} path m"] > 300).sum()),
               "start end (candidate)": int((samples[f"{s} leave m"] > 300).sum()),
               "hex end": int((samples[f"{s} arrive m"] > 300).sum()),
               "median start/end path m": samples[f"{s} path m"].median()}
display(pd.DataFrame(rows).round(0))
print("share > 300 m: before", f"{(samples['before path m'] > 300).mean():.1%}", "after", f"{(samples['after path m'] > 300).mean():.1%}")
print(f"max |before route min - table min|: {(samples['before min'] - samples['table min before']).abs().max():.3f}; "
      f"pairs where after is slower than before: {int((samples['after min'] > samples['before min'] + 0.01).sum())}")
print(f"still > 300 m after: {int((samples['after path m'] > 300).sum())}; of them, fixed in minutes (saved > 0.01): "
      f"{int(((samples['after path m'] > 300) & (samples['before min'] - samples['after min'] > 0.01)).sum())}")

Results from the run on 2026-10-29 (copied here because outputs are stripped on commit; re-running updates the printed
tables, not this one). The two-snap table built here equals `catchment-build`'s (1,187,469 pairs, same minutes).

**Full area** (res 9, X 30, 1,458 candidate sites, 4,859 hexes):

| | before (1 snap) | after (2 snaps) | change |
|---|---:|---:|---:|
| pairs within 17 min | 1,137,695 | 1,187,469 | +4.4% |
| pairs within 5.0 min (default scenario) | 50,162 | 59,855 | +19.3% |
| table on disk | 3.69 MB | 3.82 MB | +3.5% |
| `build_reach` time (OSRM already serving) | 59 s | 159 s | 2.7 times |
| `osrm-routed` peak memory (both builds, snaps and sample) | | 410 MB | |

The notebook process peaked at 446 MB (it holds both tables and the places); `catchment-build reach` itself peaked at
505 MB and took 179 s wall time, OSRM serve included. The build fits 8 GB with a wide margin: more roads per point
change the number of `/table` requests (four times as many), not the memory per request. The time varies with machine
load: a first run of this notebook measured 70 s and 192 s. The one-snap table here has 1,251 more pairs than the #13
build (1,136,444); #13 snapped candidate sites only and let OSRM snap the hex centres inside `/table`, while
`snaps=1` here snaps both ends the same way as `snaps=2`, so it is the like-for-like before.

**Second road.** 1,410 of 1,458 candidate sites (96.7%) and 3,264 of 4,859 hex centres (67.2%) got a snap point
different from the nearest road's. The rest repeat the nearest road: no second road within 100 m lying more than 5 m
across it (open land at the area's edge, a lane with nothing parallel). Probably sites sit in the built-up core, where roads run parallel, and hex centres include the sparse outskirts (not checked).

**Slice** (same slice as 0.5: 299,666 pairs before, 308,100 after; nothing in the before table dropped out):

| | |
|---|---:|
| pairs that moved (more than 0.01 min) | 225,317 of 299,666 (75.2%) |
| faster by more than 0.5 min / more than 1 min | 64,013 (21.4%) / 27,985 (9.3%) |
| minutes saved over moved pairs: median / 90th percentile / max | 0.25 / 1.18 / 11.16 |
| pairs slower | 0 |
| newly within 17 min | 8,434 |
| newly within 5.0 min | 3,711 |

Most pairs moved, but by little: the median saving is 15 s, because the fastest of four takes a few seconds off
almost any pair by choosing a better entry onto the road, not only by avoiding a far carriageway. The tail is the fix:
a tenth of the pairs gain over a minute and the largest gain is 11 min, and 3,711 pairs enter the default scenario's
5.0-minute catchments (1.2% of the slice's pairs; over the whole area the default-scenario pairs grow by 9,693). Nothing got slower, as it must when
the minimum includes the old time.

**Out-and-back on 200 short slice pairs** (before time at most 6 min, seed 42; "before" route time equals the table's
exactly, max difference 0.000 min):

| | before | after |
|---|---:|---:|
| more than 300 m at either end | 29 (14.5%) | 19 (9.5%) |
| start end (candidate site) | 14 | 6 |
| hex end | 16 | 13 |
| median start/end path | 114 m | 108 m |

(An end count adds to more than the pair count because a pair can loop at both ends.) The share drops from 14.5% to
9.5%, mainly at the candidate end (14 to 6), where the two snaps were chosen to give a divided road's two carriageways;
0.5 measured 15% (6 of 40) on a smaller sample. It is not gone. **19 pairs still ride more than 300 m at an end, and 16
of them are faster than before**: taking a better road did not remove the detour. Likely reasons, not checked here: a
one-way or a median with no opening, where both carriageways of a divided road lead the same way out and back; a second
road that is none (the nearest road repeats) or is a service road; and the hex end, where 3,264 of 4,859 centres have a
distinct second road and the rest, 1,595 hexes, only the nearest one. Whether 9.5% matters for who is reachable is for the spot-check; the
snapping row of the assumptions table already says it is a fix by geometry, not by the road's name.

## Next

The area demand map, the full-area solve and the spot-check follow in other issues of the stage 1 PRD (#9).